In [2]:
import pandas as pd


df = pd.read_csv("01 2025.csv", parse_dates=[0])
time_col = df.columns[0]
time_diff = df[time_col].diff()

df = df[(time_diff.isna()) | (time_diff >= pd.Timedelta(0))]

df["bin_start_dt"] = df[time_col].dt.floor("5min")
df["bin_time"] = df["bin_start_dt"].dt.time   # e.g. 00:00:00, 00:05:00, ...

num_cols = df.columns.difference([time_col, "bin_start_dt", "bin_time"])

df_pos = df.copy()
df_neg = df.copy()

df_pos[num_cols] = df_pos[num_cols].where(df_pos[num_cols] > 0)

df_neg[num_cols] = df_neg[num_cols].where(df_neg[num_cols] < 0)

grp_pos = df_pos.groupby("bin_time")[num_cols].mean()

grp_neg = df_neg.groupby("bin_time")[num_cols].mean()

grp_pos = grp_pos.add_suffix("_pos")
grp_neg = grp_neg.add_suffix("_neg")

result = pd.concat([grp_pos, grp_neg], axis=1)
result = result.reset_index()  # 'bin_time' is now a column

dummy_date = "2000-01-01 "
start_dt = pd.to_datetime(dummy_date + result["bin_time"].astype(str))

end_dt = start_dt + pd.Timedelta(minutes=5)

interval_str = (
    start_dt.dt.strftime("%H:%M:%S") + "-" + end_dt.dt.strftime("%H:%M:%S")
)

result.insert(0, "interval_5min", interval_str)


result = result.drop(columns=["bin_time"])

# Optionally sort by start time (should already be sorted)
result = result.sort_values("interval_5min")

fixed_cols = ["interval_5min"]

# All other columns except interval
other_cols = [c for c in result.columns if c not in fixed_cols]

base_set = {c.rsplit("_", 1)[0] for c in other_cols}

base_with_dates = [
    (b, pd.to_datetime(b, dayfirst=False))  # change to True if needed
    for b in base_set
]

base_names = [b for b, d in sorted(base_with_dates, key=lambda x: x[1])]

ordered_cols = []
for base in base_names:
    pos = base + "_pos"
    neg = base + "_neg"
    if pos in result.columns:
        ordered_cols.append(pos)
    if neg in result.columns:
        ordered_cols.append(neg)

result = result[fixed_cols + ordered_cols]

result.to_csv("01_2025_avg_5min_pos_neg.csv", index=False)
print("Saved to 01_2025_avg_5min_pos_neg.csv")

C:\Users\hsofi\AppData\Local\Temp\ipykernel_14844\1570440380.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv("01 2025.csv", parse_dates=[0])


Saved to 01_2025_avg_5min_pos_neg.csv
